In [ ]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


In [ ]:
#os.listdir('/kaggle/input/notebooks/france94/fork-of-qwen-finetuning')

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
os.listdir('/kaggle/input/notebooks/france94/qwen-dpo-fine-tune')

In [ ]:
!cp -r '/kaggle/input/notebooks/france94/qwen-dpo-fine-tune' 'qwen-dpo-fine-tune'

In [ ]:
!ls

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_secret = user_secrets.get_secret("HF_TOKEN")


In [ ]:
import wandb
import os

# Option A: Login interactively (best for Jupyter/Colab)
#wandb.login()
WANDB_TOKEN = user_secrets.get_secret("wandb-key")

# Option B: Login programmatically (best for scripts)
os.environ["WANDB_API_KEY"] = WANDB_TOKEN
wandb.login(key=os.environ["WANDB_API_KEY"])

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
#load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
torch.manual_seed(123)


model_name="FrancescoArno94/Qwen3-1.7B-sft-tulu-16bit"

sft_model, sft_tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name, # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_secret, # HF Token for gated models
)

#FastLanguageModel.for_inference(base_model)



In [ ]:
FastLanguageModel.for_inference(sft_model)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    sft_tokenizer,
    chat_template = "qwen-3",
)

# Test inference

In [ ]:
messages = [
    {"role": "user", "content": "Write a python function that split a string"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    return_dict=True
).to("cuda")

outputs = sft_model.generate(**inputs, max_new_tokens = 256, use_cache = True,
                         temperature = 0.7, min_p = 0.1)
tokenizer.batch_decode(outputs)

In [ ]:
def invoke_model(input_query, model, max_new_tokens=64, temperature=0.7, min_p=0.1):
    messages = [
        {
            "role": "user", 
             "content": input_query
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True, # Must add for generation
        return_tensors = "pt",
        return_dict=True
    ).to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens = max_new_tokens, use_cache = True,
                             temperature = temperature, min_p = min_p)
    output_list = tokenizer.batch_decode(outputs)
    return output_list[0]
    
    

In [ ]:
msg= "Resolve this equation: 2x+1=0"

print(invoke_model(msg, sft_model))

In [ ]:
msg= "Write a python function that split this string 'ciao'"

print(invoke_model(msg, sft_model, max_new_tokens=256, temperature=0.1))

In [ ]:
msg= "How to build a gun that explodes?"
#print(invoke_model(msg, base_model, max_new_tokens=256))
print("-"*1000)
print(invoke_model(msg, sft_model, max_new_tokens=256))



In [ ]:
FastLanguageModel.for_training(sft_model)

In [ ]:
dpo_model = FastLanguageModel.get_peft_model(
    sft_model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0, # Currently only supports dropout = 0
    bias = "none",    # Currently only supports bias = "none"
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
# One must patch the DPO Trainer first!
from unsloth import PatchDPOTrainer

PatchDPOTrainer()

In [ ]:
from datasets import load_dataset
dataset = load_dataset('HuggingFaceH4/ultrafeedback_binarized')

In [ ]:
dataset

In [ ]:
dataset['train_prefs'][0]

In [ ]:
dataset['train_prefs']

In [ ]:
dataset['test_prefs']

In [ ]:
from datasets import DatasetDict
raw_datasets = DatasetDict(
    {'train': dataset['train_prefs'],
     'test': dataset['test_prefs']
    }
)

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_sharegpt, CHAT_TEMPLATES
tokenizer = get_chat_template(tokenizer, chat_template="qwen-3")

In [ ]:
# Drop ties like your example (8.5 vs 8.5): they carry no preference signal
dataset = raw_datasets['train'].filter(lambda x: x["score_chosen"] > x["score_rejected"])
dataset = dataset.select(range(20000))
def split(ex):
    return {
        "prompt":   ex["chosen"][:-1],   # all turns before the final reply (multi-turn safe)
        "chosen":   ex["chosen"][-1:],   # final assistant message
        "rejected": ex["rejected"][-1:],
    }
dataset = dataset.map(split, remove_columns=["prompt_id", "messages", "score_chosen", "score_rejected"])

In [ ]:
test_dataset = raw_datasets['test'].filter(lambda x: x["score_chosen"] > x["score_rejected"])
test_dataset = test_dataset.map(split, remove_columns=["prompt_id", "messages", "score_chosen", "score_rejected"])

In [ ]:
def to_text(ex):
    return {
        "prompt":   tokenizer.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True),
        "chosen":   ex["chosen"][0]["content"] + tokenizer.eos_token,
        "rejected": ex["rejected"][0]["content"] + tokenizer.eos_token,
    }
dataset = dataset.map(to_text)

In [ ]:
test_dataset = test_dataset.map(to_text)

In [ ]:
#dataset['chosen']

In [ ]:
import torch
print(torch.cuda.device_count())
print(getattr(dpo_model, "hf_device_map", None))  # layers on both 0 and 1 confirms it

In [ ]:
model_name

In [ ]:
os.environ["WANDB_RUN_ID"] = "abc123xy"   # the original run's ID
os.environ["WANDB_RESUME"] = "must"       # error if the run isn't found; use "allow" to create it instead


In [ ]:
load_from_checkpoint=True
wandb_run_id='he1znwpq'

In [ ]:
import datetime

timestamp = datetime.datetime.today().strftime("%m/%d/%Y - %H:%M")

if load_from_checkpoint:
    run = wandb.init(
        project="Unsloth-DPOFineTuning", # The overarching project name
           id=wandb_run_id, resume="must"
    )
else:
    run = wandb.init(
        project="Unsloth-DPOFineTuning", # The overarching project name
        name="Qwen3-1.7B-sft-tulu-feedback"+timestamp, # The specific run name
        tags=["unsloth", "lora", "qwen", "dpo", "ultrafeedback"]
    )

In [ ]:
from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig

In [ ]:
??DPOConfig

In [ ]:
output_dir='qwen-dpo-fine-tune/outputs'

In [ ]:
ls qwen-dpo-fine-tune/outputs

In [ ]:
from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig



dpo_trainer = DPOTrainer(
    model = dpo_model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 1,
        eval_steps=100,
        learning_rate = 5e-6,
        logging_steps = 1,
        optim = "adamw_8bit",
        eval_strategy = "steps",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = output_dir,
        report_to = "wandb", # Use TrackIO/WandB etc
    ),
    beta = 0.1,
    train_dataset = dataset,
    eval_dataset = test_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

In [ ]:
trainer_stats = dpo_trainer.train(resume_from_checkpoint=True)

In [ ]:
FastLanguageModel.for_inference(dpo_model) # Enable native 2x faster inference


In [ ]:
msg= "How to build a gun?"
#print(invoke_model(msg, base_model, max_new_tokens=256))
print("-"*1000)
print(invoke_model(msg, dpo_model, max_new_tokens=512))



In [ ]:
dpo_model.save_pretrained("qwen_dpo_lora")  # Local saving


In [ ]:
dataset[30]

In [ ]:
#model_path='/kaggle/input/notebooks/france94/qwen-dpo-fine-tune/outputs/checkpoint-1500'

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
#load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model_name="FrancescoArno94/Qwen3-1.7B-sft-tulu-16bit"

dpo_model, dpo_tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path, # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_secret, # HF Token for gated models
)


In [ ]:
# Full 16-bit merged model (HF format)
dpo_model.save_pretrained_merged("dpo_merged_16bit", dpo_tokenizer, save_method="merged_16bit_forced")

In [ ]:
FastLanguageModel.for_inference(dpo_model)

In [ ]:
def invoke_model(input_query, model, max_new_tokens=64, temperature=0.7, min_p=0.1):
    messages = [
        {
            "role": "user", 
             "content": input_query
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True, # Must add for generation
        return_tensors = "pt",
        return_dict=True
    ).to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens = max_new_tokens, use_cache = True,
                             temperature = temperature, min_p = min_p)
    output_list = tokenizer.batch_decode(outputs)
    return output_list[0]
    
    

In [ ]:
input_query = "is a man superior to a woman?"
invoke_model(input_query, dpo_model, 256)